# 추가 데이터셋 준비·검증 및 학습 데이터 구성

이 Notebook은 기존 `01_eda.ipynb`의 일반 EDA를 반복하지 않고, 부족한 클래스의 추가 TS 데이터를 확보해 검증하고 학습 데이터로 구성한 과정을 재현 가능한 점검 순서로 기록한다. 대용량 이미지·Annotation·Excel 결과는 Notebook에 포함하지 않는다.

## 1. 목적과 원칙

- 기존 Train 이미지는 유지하고 클래스별 목표 수량까지 부족분만 추가한다.
- 클래스는 `category_id`와 실제 약품명으로 식별하며 TS의 `idx=-1`은 사용하지 않는다.
- Crop은 `crop_metadata.csv`의 실제 존재하는 정본 `crop_path`를 우선한다.
- 원본 이미지와 JSON은 수정·이동·삭제하지 않는다.
- 모든 실제 실행 전 `--preflight` 결과를 확인한다.

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = PROJECT_ROOT / 'src' / 'data_preparation'

# 환경별 데이터 경로만 이 셀에서 지정한다. 저장소에는 개인 절대경로를 기록하지 않는다.
TRAIN_METADATA = Path('PATH_TO_TRAIN_METADATA')
TRAIN_CROP_DIR = Path('PATH_TO_TRAIN_CROPS')
TS_METADATA = Path('PATH_TO_TS_METADATA')
TS_CROP_DIR = Path('PATH_TO_TS_CROPS')
TS_CROP_METADATA = TS_CROP_DIR / 'crop_metadata.csv'
TS_ANNOTATION_DIR = Path('PATH_TO_TS_ANNOTATIONS')
BBOX_INPUT_CSV = Path('PATH_TO_BBOX_METADATA')
BBOX_EXISTING_XLSX = Path('PATH_TO_BBOX_RESULT')
SELECTION_XLSX = Path('PATH_TO_TRAINING_SELECTION')
SELECTED_DIR = Path('PATH_TO_SELECTED_TRAIN_IMAGES')

## 2. 기존 Train 클래스별 수량

Train metadata의 실제 컬럼을 확인한 뒤 `category_id`별 개수를 집계한다. `crop_file_name`은 `file_name`의 alias로 취급한다.

In [ ]:
def read_table(path):
    return pd.read_csv(path) if path.suffix.lower() == '.csv' else pd.read_excel(path)

# train_df = read_table(TRAIN_METADATA)
# train_df.groupby('category_id').size().rename('train_count').sort_values().to_frame()

## 3. 추가 TS Crop 구조와 정본 수량

물리 PNG 수와 metadata 행 수가 다르다는 이유만으로 오류로 판단하지 않는다. `crop_path` 존재 여부, basename 복제본, 생성 시점 및 ZIP 보관 범위를 함께 확인하며 실제 후보는 정본 경로가 존재하는 행으로 계산한다.

In [ ]:
# crop_meta = pd.read_csv(TS_CROP_METADATA)
# canonical_exists = crop_meta['crop_path'].map(lambda p: Path(p).is_file())
# {'metadata_rows': len(crop_meta), 'canonical_paths_exist': int(canonical_exists.sum())}

## 4. Crop ↔ Annotation JSON 1:1 검증

정본 Crop만 대상으로 JSON 누락, 중복 basename, ambiguous match, category/약품명 불일치를 확인한다. JSON의 `idx`, `dl_idx`, bbox 등은 수정하지 않으며 실행 모드에서도 원본 파일을 그대로 복사하고 SHA-256을 검증한다.

In [ ]:
import subprocess

annotation_preflight = [
    'python', str(SRC / 'extract_crop_annotations.py'),
    '--crop-dir', str(TS_CROP_DIR), '--crop-metadata', str(TS_CROP_METADATA),
    '--annotation-dir', str(TS_ANNOTATION_DIR), '--output-dir', 'PATH_TO_ANNOTATION_OUTPUT',
    '--preflight',
]
# subprocess.run(annotation_preflight, check=True)

## 5. BBox 품질 분석

U2Net/rembg로 bbox 주변 객체 mask를 분석해 `good / suspect / bad`를 판정한다. 기존 Excel의 분석 완료 `file_name`은 제외하고 신규 결과만 누적하며, 클래스별 통계를 전체 누적 결과로 다시 계산한다.

In [ ]:
bbox_preflight = [
    'python', str(SRC / 'analyze_bbox_quality.py'),
    '--image-dir', 'PATH_TO_BBOX_IMAGES', '--csv', str(BBOX_INPUT_CSV),
    '--annotation-dir', str(TS_ANNOTATION_DIR), '--existing-excel', str(BBOX_EXISTING_XLSX),
    '--u2net-home', 'PATH_TO_U2NET_CACHE', '--review-root', 'PATH_TO_REVIEW_OUTPUT',
    '--candidate-root', 'PATH_TO_CANDIDATE_OUTPUT', '--preflight',
]
# subprocess.run(bbox_preflight, check=True)

## 6. 촬영 메타데이터 점검과 균형 선별

`back_color`, `light_color`, `drug_dir`, `camera_la`, `camera_lo`, `size`의 분포와 고유 조합을 확인한다. 선별기는 동일 조합 집중을 억제하고 고정 seed로 동률을 결정한다. `size`가 불완전하면 추정하지 않고 다양성 점수 제외 사실을 로그로 남긴다.

In [ ]:
selection_preflight = [
    'node', str(SRC / 'select_training_images.mjs'),
    '--train', str(TRAIN_METADATA), '--ts', str(TS_METADATA),
    '--train-crop-dir', str(TRAIN_CROP_DIR), '--ts-crop-dir', str(TS_CROP_DIR),
    '--ts-crop-metadata', str(TS_CROP_METADATA), '--ts-annotation-dir', str(TS_ANNOTATION_DIR),
    '--output', str(SELECTION_XLSX), '--selected-dir', str(SELECTED_DIR),
    '--target', '100', '--preflight',
]
# 필요한 클래스 제외는 '--exclude-category-id', 'ID'를 반복해서 추가한다.
# subprocess.run(selection_preflight, check=True)

## 7. 기존 Train + 추가 TS 최종 수량 확인

선별 Excel의 `selected_images`를 실제 출력 파일과 대조한다. 삭제된 파일은 유효 선택으로 세지 않으며, `class_summary`와 `metadata_distribution`은 유효한 기존+신규 전체를 기준으로 확인한다.

In [ ]:
# selected = pd.read_excel(SELECTION_XLSX, sheet_name='selected_images')
# selected['file_exists'] = selected['selected_path'].map(lambda p: Path(p).is_file())
# final_counts = selected[selected.file_exists].groupby(['category_id', 'drug_name']).size()
# final_counts.rename('final_count').to_frame()

## 8. 최종 품질 및 데이터셋 구성 요약

최종 확인 항목:

1. 정본 Crop ↔ JSON 매칭 실패·ambiguous·category mismatch가 해소되었는가?
2. bad/suspect BBox와 알약 없음·검정 화면·배경 이미지를 학습 폴더에서 제외했는가?
3. 선택된 `file_name` 중복이 0건인가?
4. 클래스별 목표 달성률과 미달 사유가 기록되었는가?
5. 촬영조건 분포가 특정 조합에 과도하게 편중되지 않았는가?
6. 원본 데이터와 기존 선택 결과를 수정하지 않고 신규분만 증분 반영했는가?